In [218]:
import pandas as pd
import numpy as np
from IPython.display import Math

In [219]:
training_set = pd.read_csv('/home/lukas/Desktop/digit-recognizer/train.csv')
test = pd.read_csv('/home/lukas/Desktop/digit-recognizer/test.csv')

In [220]:
train_set = training_set.to_numpy()
y = training_set['label'].to_numpy()

In [284]:
def create_batches(arr, batch_size):
    rng.shuffle(arr)
    container = []
    labels = []
    
    for i in range(0, len(arr), batch_size):
        sample = arr[i : i + batch_size, 1:]
        sample = sample / 255.0
        label = arr[i : i + batch_size, 0]
        labels.append(label)
        container.append(sample)

    return container, labels

In [222]:
def ReLU(x):
    x = np.maximum(0, x)
    return x

In [223]:
def Softmax(x):
    
    x = x - np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x)
    output = exp_x / np.sum(exp_x, axis=1, keepdims=True)

    return output

In [224]:
def layer(weights, bias, x_in, func):
    
    z = x_in @ weights + bias
    activation = func(z)

    return z, activation

In [225]:
def init_parameters(layer_neurons):
    num_layers = len(layer_neurons)
    list_weights = []
    list_bias = []
    
    for i in range(1, num_layers):
        n_in = layer_neurons[i - 1]
        n_out = layer_neurons[i]

        weight = np.random.randn(n_in, n_out) * np.sqrt(2 / n_in)
        list_weights.append(weight)
        list_bias.append(np.zeros((1, n_out)))

    return list_weights, list_bias

In [226]:
def forward_prop(x_in, w, b):
    num_layers = len(w)
    activation_dict = {}
    z_dict = {}
    
    for i in range(num_layers - 1):
        z, a = layer(w[i], b[i], x_in, ReLU)
        x_in = a
        z_dict[i + 1] = z
        activation_dict[i + 1] = a
        
    z, a = layer(w[-1], b[-1], z, Softmax)
    z_dict[num_layers] = z
    activation_dict[num_layers] = a
    
    return activation_dict, z_dict

In [342]:
ad, zd = forward_prop(x_train[0], w, b)

In [344]:
ad[3].shape

(32, 10)

In [227]:
def compute_cost(softmax_vector, labels):
    eps = 1e-15
    rows = np.arange(len(labels))
    cols = labels
    x = softmax_vector[rows, cols]
    
    loss = np.mean(-np.log(x + eps))
    return loss

In [228]:
def ReLU_derivative(z):
    return (z > 0).astype(float)

In [325]:
def backpropagation(w, b, activation_dict, z_dict, y_in, x_in):

    # One-Hot Encoder
    aha, ehe = activation_dict[3].shape
    error_vector = np.zeros((aha, ehe))
    rows = np.arange(aha)
    cols = y_in
    error_vector[rows, cols] = 1
    
    num_operations = len(w)
    gradients_w = [None] * num_operations
    gradients_b = [None] * num_operations

    ###  Pochodna dC/dw
    A_out = activation_dict[num_operations] ### Funkcja aktywacji którą wyrzuca output layer
    A_prev = activation_dict[num_operations - 1] ### Funkcja aktywacji która trafia do output layer

    # dC/da(L)
    gamma = A_out - error_vector

    gradients_w[-1] = A_prev.T @ gamma
    gradients_b[-1] = np.sum(gamma, axis = 0, keepdims=True)

    for L in reversed(range(1, num_operations)):

        if L == 1:
            A_prev = x_in
        else:
            A_prev = activation_dict[L]

        # gamma z następnej warstwy * waga następnej warstwy * pochodna funkcji aktywacji aktualnej warstwy
        gamma = (gamma @ w[L].T) * ReLU_derivative(z_dict[L])
        gradients_w[L-1] = A_prev.T @ gamma
        gradients_b[L-1] = np.sum(gamma, axis = 0, keepdims=True)


    return gradients_w, gradients_b
    

In [326]:
rng = np.random.default_rng()

In [356]:
layer_neurons = [784, 10, 10, 10]
w, b = init_parameters(layer_neurons)

In [361]:
epochs = 101
learning_rate = 0.0001

In [360]:
for epoch in range(epochs):
    
    x_train, y_train = create_batches(train_set, 32)
    
    for batch_idx in range(len(x_train)):
        a_dict, z_dict = forward_prop(x_train[batch_idx], w, b)
        gradients_w, gradients_b = backpropagation(w, b, a_dict, z_dict, y_train[batch_idx], x_train[batch_idx])

        for i in range(len(gradients_w)):
            w[i] = w[i] - learning_rate * gradients_w[i]
            b[i] = b[i] - learning_rate * gradients_b[i]

    if epoch % 20 == 0:
        loss = compute_cost(a_dict[3], y_train[batch_idx])
        print(f"Koszt w epoce nr: {epoch}, wyniósł: {loss}")
                            

Koszt w epoce nr: 0, wyniósł: 0.3487392279242071
Koszt w epoce nr: 20, wyniósł: 0.23623442665908753
Koszt w epoce nr: 40, wyniósł: 0.4464028552958156
Koszt w epoce nr: 60, wyniósł: 0.4195529798414313
Koszt w epoce nr: 80, wyniósł: 0.6321351581995146
Koszt w epoce nr: 100, wyniósł: 0.09359000806049199


In [ ]:
tg